# GraphSynth as a CHIA loop

Two ways to run this notebook.

**Replay (no GPU needed).** Executes the loop's graph, tools and database for
real, with node bodies replaced by the recorded A100 run via CHIA's `Bypass`.
Finishes in about a minute and reproduces the paper's 10/10 at 10.83x.

**Live.** Needs a bf16-capable GPU (Runtime > Change runtime type: A100 or L4 —
a T4 is sm75 and cannot run the recorded bf16 kernels) and Vertex AI
credentials.


In [ ]:
!pip install -q chialoops
!git clone -q https://github.com/JoshiP-commits/CHIA_proj.git
%cd CHIA_proj

## Replay — no GPU, no credentials

In [ ]:
!python -m chia_loop.loop --bypass chia_loop/configs/bypass_replay.yaml --db /content/attempts.db

## Inspect the attempts database the loop filled

In [ ]:
import sqlite3, pandas as pd
c = sqlite3.connect("/content/attempts.db")
display(pd.read_sql("SELECT op, iteration, passed, rel_err, kernel_us, speedup FROM attempts", c))
display(pd.read_sql("SELECT op, ai, ridge, memory_bound, admitted FROM profile", c))

## Live run

Check the GPU first: bf16 needs compute capability 8.0 or higher.

In [ ]:
import torch
print(torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))
print("bf16 usable:", torch.cuda.get_device_capability(0)[0] >= 8)

In [ ]:
# Requires Vertex AI credentials.
# from google.colab import auth; auth.authenticate_user()
# !GCP_PROJECT=<your-project> python -m chia_loop.loop --ops bias_block_diagonal --project <your-project>

On an sm75 GPU such as a T4, run a reduced fp16 shape instead. It
demonstrates the loop, not the paper's numbers:

```
!python -m chia_loop.loop -B 1 -H 8 -S 512 -D 64 --dtype float16 --ops bias_block_diagonal
```